## Parte 1: Importaciones de bibliotecas

Antes de correr cualquier código, necesitamos traer al proyecto las herramientas externas que vamos a usar. Cada línea de `import` es como instalar y activar una herramienta específica:

- **`io`** — Permite trabajar con datos en memoria como si fueran archivos, sin necesidad de escribirlos al disco primero.
- **`os`** — Permite interactuar con el sistema operativo: crear carpetas, construir rutas de archivos, verificar si un directorio existe.
- **`datetime`** — Maneja fechas y horas. Se usa para registrar cuándo se creó o procesó un archivo.
- **`hashlib`** — Genera una "huella digital" única (firma MD5) para cada imagen. Dos imágenes idénticas tendrán la misma firma; cualquier diferencia producirá una firma completamente distinta.
- **`fitz` (PyMuPDF)** — Abre y procesa archivos PDF. Permite leer el contenido, obtener el número de páginas y renderizar cada página como una imagen.
- **`pydantic`** — Valida y estructura los datos. Garantiza que los objetos que creamos tengan exactamente los campos que esperamos, con los tipos correctos.
- **`pytesseract`** — Motor de reconocimiento óptico de caracteres (OCR). Analiza una imagen y extrae el texto que aparece en ella.
- **`PIL / Image`** — Librería para abrir, manipular y guardar imágenes en múltiples formatos (PNG, JPEG, etc.).

> Las dependencias con sus versiones exactas están en `assets/requirements.txt`.

In [25]:
import io
import os
import datetime
import hashlib

import fitz
import pydantic
import pytesseract
from PIL import Image

## Parte 2: Definición de estructuras de datos

Aquí definimos los "moldes" o plantillas que describen cómo se ve la información que vamos a manejar. Cada clase es como un formulario con campos predefinidos; cuando creamos un objeto de esa clase, estamos llenando ese formulario.

---

### `PDFMetadata`
Guarda toda la información relevante sobre el archivo PDF que se va a procesar:

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `file_name` | texto | Nombre del archivo (ej. `contrato.pdf`) |
| `file_data` | bytes | El contenido binario completo del PDF |
| `file_size` | número | Peso del archivo en bytes |
| `pages_num` | número | Cantidad de páginas que tiene el PDF |
| `upload_at` | fecha/hora | Momento en que se cargó el archivo |

---

### `ImageMetadata`
Guarda la información de cada imagen generada a partir de una página del PDF:

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `file_name` | texto | Nombre de la imagen (ej. `contrato_page_1.png`) |
| `file_data` | bytes | El contenido binario de la imagen |
| `file_size` | número | Peso de la imagen en bytes |
| `page_number` | número | Número de página del PDF que representa |
| `image_path` | texto | Ruta completa donde se guardó en disco |
| `created_at` | fecha/hora | Momento en que se generó la imagen |

---

### `ProcesorData`
Guarda el resultado final del procesamiento OCR para cada imagen:

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `image_name` | texto | Nombre de la imagen procesada |
| `signature` | texto | Firma MD5 única de la imagen (32 caracteres hexadecimales) |
| `text` | texto | Texto extraído de la imagen mediante OCR |

In [26]:
from datetime import datetime as dt
from pydantic import BaseModel


class PDFMetadata(BaseModel):
    file_name: str
    file_data: bytes
    file_size: int
    pages_num: int
    upload_at: dt = dt.now

class ImageMetadata(BaseModel):
    file_name: str
    file_data: bytes
    file_size: int
    page_number: int
    image_path: str = ""
    created_at: dt = dt.now

class ProcesorData(BaseModel):
    image_name: str
    signature: str
    text: str

## Parte 3: Procesador OCR — lectura del PDF, extracción de imágenes y texto

Esta sección contiene toda la lógica del proceso. La clase `OCRProcessor` agrupa los pasos necesarios para convertir un PDF en texto plano. Funciona como una línea de ensamblaje: cada método realiza una tarea específica y pasa el resultado al siguiente.

---

### Clase `OCRProcessor`

#### `open_pdf(pdf_route)` — Leer el PDF
Recibe la ruta del archivo PDF en disco, lo abre y devuelve un objeto `PDFMetadata` con su nombre, contenido binario, tamaño y número de páginas. Es el punto de entrada de todo el proceso.

#### `separete_images(pdf_metadata)` — Convertir páginas a imágenes
Toma el PDF (en memoria) y renderiza cada página como una imagen PNG a 200 DPI. Cada imagen se guarda temporalmente en memoria antes de pasarla al siguiente paso. Devuelve una lista de objetos `ImageMetadata`, uno por página.

#### `save_images(images, save_route)` — Guardar las imágenes en disco
Recibe la lista de imágenes y una carpeta de destino. Crea la carpeta si no existe y escribe cada imagen como archivo `.png`. Actualiza la ruta en cada `ImageMetadata` para que los pasos posteriores sepan dónde encontrar cada archivo.

#### `extract_text(images)` — Extraer texto con OCR
Para cada imagen: genera su firma MD5, la abre con PIL y la pasa por `pytesseract` para extraer el texto. Devuelve una lista de objetos `ProcesorData` con el nombre de imagen, firma y texto extraído.

#### `initialize_process_exttraction(pdf_route, save_images_route)` — Orquestar el proceso completo
Método principal que encadena todos los pasos anteriores en orden:
1. Abre el PDF
2. Convierte sus páginas a imágenes
3. Guarda las imágenes en disco
4. Extrae el texto de cada imagen

Imprime mensajes de progreso en cada etapa y devuelve la lista final de `ProcesorData`.

---

### Función `save_text_to_file(info_images, output_path, signature_num)`
Función independiente que guarda el texto extraído en archivos `.txt` separados, uno por página. Los archivos se nombran siguiendo el patrón `signature{N}_{i}.txt`, donde `N` es el número de firma del lote y `i` es el número de página (comenzando en 1).

**Ejemplo:** si se procesan 3 páginas con `signature_num=1`, se crean:
- `signature1_1.txt`
- `signature1_2.txt`
- `signature1_3.txt`

> Para cambiar el PDF a procesar, modifica la variable `ROUTE_PDF` en la siguiente sección.

In [27]:
class OCRProcessor:
    def open_pdf(self, pdf_route: str) -> PDFMetadata:
        with open(pdf_route, "rb") as f:
            file_data = f.read()
        doc = fitz.open(pdf_route)
        return PDFMetadata(
            file_name=os.path.basename(pdf_route),
            file_data=file_data,
            file_size=len(file_data),
            pages_num=len(doc),
        )

    def separete_images(self, pdf_metadata: PDFMetadata) -> list[ImageMetadata]:
        doc = fitz.open(stream=pdf_metadata.file_data, filetype="pdf")
        images = []
        for i, page in enumerate(doc):
            pix = page.get_pixmap(dpi=200)
            img_bytes = pix.tobytes("png")
            images.append(ImageMetadata(
                file_name=f"{os.path.splitext(pdf_metadata.file_name)[0]}_page_{i + 1}.png",
                file_data=img_bytes,
                file_size=len(img_bytes),
                page_number=i + 1,
            ))
        return images

    def save_images(self, images: list[ImageMetadata], save_route: str) -> list[ImageMetadata]:
        os.makedirs(save_route, exist_ok=True)
        updated_images = []
        for img_meta in images:
            path = os.path.join(save_route, img_meta.file_name)
            with open(path, "wb") as f:
                f.write(img_meta.file_data)
            img_meta.image_path = path
            updated_images.append(img_meta)
        return updated_images

    def extract_text(self, images: list[ImageMetadata]) -> list[ProcesorData]:
        procesor_data_list = []
        for img_meta in images:
            signature = hashlib.md5(img_meta.file_data).hexdigest()
            pil_img = Image.open(img_meta.image_path)
            text = pytesseract.image_to_string(pil_img)
            procesor_data_list.append(ProcesorData(
                image_name=img_meta.file_name,
                signature=signature,
                text=text,
            ))
        return procesor_data_list

    def initialize_process_exttraction(self, pdf_route: str, save_images_route: str) -> list[ProcesorData]:
        pdf_metadata = self.open_pdf(pdf_route)
        print(f"PDF abierto: {pdf_metadata.file_name} ({pdf_metadata.pages_num} páginas)")

        images = self.separete_images(pdf_metadata)
        print(f"Páginas convertidas a imagen: {len(images)}")

        new_images = self.save_images(images, save_images_route)
        print(f"Imágenes guardadas en: {save_images_route}")

        ocr_info = self.extract_text(new_images)
        print("Extracción de texto completada.")
        return ocr_info


def save_text_to_file(data: ProcesorData, output_path: str):
    os.makedirs(output_path, exist_ok=True)
    file_name = f"{data.signature}.txt"
    file_path = os.path.join(output_path, file_name)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(data.text)
    print(f"Texto guardado en: {file_path}")

## Parte 4: Ejecución del proceso

Esta sección es el punto de arranque. Aquí se configuran las rutas de los archivos y se lanza el proceso completo de extracción.

### Variables de configuración

| Variable | Descripción |
|----------|-------------|
| `ROUTE_PDF` | Ruta completa al archivo PDF que se quiere procesar. **Cambia esta ruta para apuntar a tu propio PDF.** |
| `ROUTE_SAVE_IMAGES` | Carpeta donde se guardarán las imágenes generadas por cada página del PDF. Se crea automáticamente si no existe. |
| `ROUTE_SAVE_TEXT` | Carpeta donde se guardarán los archivos `.txt` con el texto extraído. Se crea automáticamente si no existe. |

### Flujo de ejecución

1. Se crea una instancia de `OCRProcessor` (el motor de procesamiento).
2. Se llama a `initialize_process_exttraction` con la ruta del PDF y la carpeta de imágenes. Este método ejecuta automáticamente todos los pasos internos (leer PDF → convertir a imágenes → guardar imágenes → extraer texto).
3. Se llama a `save_text_to_file` para guardar el texto de cada página en un archivo `.txt` numerado dentro de la carpeta de texto.
4. Se imprime un resumen por cada página procesada: nombre de imagen, los primeros 8 caracteres de la firma y los primeros 80 caracteres del texto extraído.

In [29]:
ROUTE_PDF = "/Users/jmonroy/Downloads/CV_JACOBO_MONROY_2026.pdf"
ROUTE_SAVE_IMAGES = "/Users/jmonroy/Documents/MyProjects/NeuralBank/notebooks/images"
ROUTE_SAVE_TEXT = "/Users/jmonroy/Documents/MyProjects/NeuralBank/notebooks/text"

if __name__ == "__main__":
    ocr_processor = OCRProcessor()
    info = ocr_processor.initialize_process_exttraction(ROUTE_PDF, ROUTE_SAVE_IMAGES)
    for data in info:
        save_text_to_file(data, ROUTE_SAVE_TEXT)
        print(f"Imagen: {data.image_name} | Firma: {data.signature}... | Texto: {data.text}...")

PDF abierto: CV_JACOBO_MONROY_2026.pdf (2 páginas)
Páginas convertidas a imagen: 2
Imágenes guardadas en: /Users/jmonroy/Documents/MyProjects/NeuralBank/notebooks/images
Extracción de texto completada.
Texto guardado en: /Users/jmonroy/Documents/MyProjects/NeuralBank/notebooks/text/15a154c6acbcd5f6ee6548a7ea007e43.txt
Imagen: CV_JACOBO_MONROY_2026_page_1.png | Firma: 15a154c6acbcd5f6ee6548a7ea007e43... | Texto: Jacobo Monroy C

9 CDMX, MEXICO = jmonroy@kinasisdev.com 5587957504 @ jmonroy.kinasisdev.com J monroy-jacobo

Q killahblitz
About Me

Computer Engineer (IPN) and Full-Stack Developer with production experience designing containerized Python microservices,
serverless cloud architectures on AWS, and reactive interfaces using Vue.js (Nuxt) and React. Experienced as a Tech Lead managing
end-to-end CI/CD pipelines and cross-functional teams under Agile methodologies. Specialized in building scalable, data-driven
systems integrating Machine Learning and LLM-driven structured data extr